# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [3]:
# Desarrollo de la actividad 1
total_eva = len(eva)
print("Dimensiones:", eva.shape, "| ¿166.732 filas?", total_eva == 166732)
print(eva.columns.tolist())
display(eva.head(3))

codigo = eva["Código Dane departamento"].str.strip()
eva["codigo_departamento"] = codigo.where(codigo.str.fullmatch(r"[0-9]{1,2}"), pd.NA).str.zfill(2)
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip(), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip(), errors="coerce")

print(eva[["codigo_departamento", "anio", "area_ha", "produccion_t"]].dtypes)

origen = {
    "codigo_departamento": "Código Dane departamento",
    "anio": "Año",
    "area_ha": "Área cosechada",
    "produccion_t": "Producción",
}
ausencias = pd.DataFrame([
    {
        "columna": nueva,
        "entrada_vacia": int(eva[original].str.strip().eq("").sum()),
        "no_convertible": int((eva[original].str.strip().ne("") & eva[nueva].isna()).sum()),
        "total_ausente": int(eva[nueva].isna().sum()),
    }
    for nueva, original in origen.items()
]).set_index("columna")
display(ausencias)

for nueva, original in origen.items():
    malos = eva.loc[eva[original].str.strip().ne("") & eva[nueva].isna(), original]
    if len(malos):
        print(f"Valores no convertibles en {original}:", malos.value_counts().head(5).to_dict())

print("Años:", {str(k): int(v) for k, v in eva["anio"].value_counts(dropna=False).sort_index().items()})
assert len(eva) == total_eva

Dimensiones: (166732, 18) | ¿166.732 filas? True
['Código Dane departamento', 'Departamento', 'Código Dane municipio', 'Municipio', 'Grupo cultivo', 'Subgrupo', 'Cultivo', 'Desagregación cultivo', 'Año', 'Periodo', 'Área sembrada', 'Área cosechada', 'Producción', 'Rendimiento', 'Ciclo del cultivo', 'Estado físico del cultivo', 'Código del cultivo', 'Nombre científico del cultivo']


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo
0,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021A,8.00,8.00,128.00,16.00,Transitorio,En fresco,1050600,Apium graveolens
1,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021B,8.00,8.00,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
2,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022A,8.00,8.00,127.44,15.93,Transitorio,En fresco,1050600,Apium graveolens


codigo_departamento     object
anio                     Int64
area_ha                float64
produccion_t           float64
dtype: object


,entrada_vacia,no_convertible,total_ausente
columna,,,
codigo_departamento,0,0,0
anio,0,0,0
area_ha,0,0,0
produccion_t,0,0,0


Años: {'2019': 20436, '2020': 21511, '2021': 23898, '2022': 24808, '2023': 24909, '2024': 25553, '2025': 25617}


**Interpretación del resultado:**

EVA tiene 166.732 registros y 18 columnas, igual que el corte del manifiesto, y no se eliminó ninguna fila. Cada fila es una observación de la Evaluación Agropecuaria: un cultivo, en un municipio, en un periodo (por ejemplo, Apio en Medellín, periodo 2021A), con su área sembrada, área cosechada, producción y rendimiento. Una fila no es un predio ni un productor.

Se crearon cuatro columnas de trabajo y se conservaron las originales:

codigo_departamento sigue siendo texto de dos cifras ("05" para Antioquia, "15" para Boyacá), porque es un código y no una cantidad. Convertirlo a número haría perder el cero inicial ("05" pasaría a 5) y ya no coincidiría con otras tablas.

anio es Int64 (entero que admite ausencias), y area_ha y produccion_t son float64.

La tabla de ausencias muestra 0 entradas vacías y 0 valores no convertibles en las cuatro columnas: todos los códigos, años, áreas y producciones son interpretables. Que no haya ausencias no significa que todas las filas sirvan para calcular indicadores, porque puede haber áreas cosechadas iguales a 0 (cultivos sembrados que aún no se cosechan). Eso se controla en la actividad 2 con la condición de área > 0.

Los registros cubren de 2019 a 2025 y crecen de forma sostenida: 20.436 (2019), 21.511 (2020), 23.898 (2021), 24.808 (2022), 24.909 (2023), 25.553 (2024) y 25.617 (2025). El aumento refleja más observaciones reportadas por año, no necesariamente más área cultivada.

## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [4]:
# Desarrollo de la actividad 2
cond_departamento = eva["codigo_departamento"].eq("15")
cond_anio = eva["anio"].isin([2023, 2024])
cond_area = eva["area_ha"].gt(0)
cond_produccion = eva["produccion_t"].ge(0)

mascara = (cond_departamento & cond_anio & cond_area & cond_produccion).fillna(False).astype(bool)
consulta_loc = eva.loc[mascara].copy()

# Cada fila seleccionada cumple las cuatro condiciones a la vez.
assert consulta_loc["codigo_departamento"].eq("15").all()
assert consulta_loc["anio"].isin([2023, 2024]).all()
assert consulta_loc["area_ha"].gt(0).all()
assert consulta_loc["produccion_t"].ge(0).all()

conteos_seleccion = pd.Series({
    "total_original": total_eva,
    "seleccionado": len(consulta_loc),
    "diferencia": total_eva - len(consulta_loc),
}, name="filas")
display(conteos_seleccion.to_frame())

# Desglose: cuántas filas de Boyacá 2023-2024 se pierden por área o producción.
boyaca_periodo = (cond_departamento & cond_anio).fillna(False).astype(bool)
print("Filas Boyacá 2023-2024 (sin filtros de medida):", int(boyaca_periodo.sum()))
print("  excluidas por área no positiva o ausente:", int((boyaca_periodo & ~cond_area.fillna(False)).sum()))
print("  excluidas por producción negativa o ausente:", int((boyaca_periodo & ~cond_produccion.fillna(False)).sum()))
print("Selección por año:", {int(k): int(v) for k, v in consulta_loc["anio"].value_counts().sort_index().items()})

,filas
total_original,166732
seleccionado,4924
diferencia,161808


Filas Boyacá 2023-2024 (sin filtros de medida): 5318
  excluidas por área no positiva o ausente: 394
  excluidas por producción negativa o ausente: 0
Selección por año: {2023: 2436, 2024: 2488}


**Interpretación del resultado:**

`consulta_loc` combina cuatro condiciones con `&`: departamento `"15"` (Boyacá, comparado como texto), año 2023 o 2024, área cosechada > 0 y producción ≥ 0. Los `assert` confirman que **todas** las filas seleccionadas cumplen las cuatro condiciones a la vez.

| Concepto | Filas |
|---|---:|
| Total original | 166.732 |
| Seleccionado (`consulta_loc`) | **4.924** |
| Diferencia | 161.808 |

La diferencia es grande, pero **casi toda se explica por el alcance de la pregunta**: 161.414 filas corresponden a otros departamentos o a otros años (2019–2022 y 2025). Esas filas no son errores; simplemente están fuera de la consulta.

Dentro de Boyacá 2023–2024 había **5.318 filas**. De ellas se excluyeron **394 (7,4 %) por área cosechada no positiva** y **0 por producción**. Como la actividad 1 mostró que no hay valores ausentes, esas 394 filas tienen área cosechada igual a 0: son cultivos reportados en el periodo pero sin cosecha registrada, por ejemplo cultivos recién sembrados o perdidos. Se cumple 5.318 − 394 = 4.924.

La selección queda equilibrada entre años: **2.436 registros en 2023 y 2.488 en 2024**.

Se usó `.fillna(False)` en la máscara para que cualquier comparación con un valor ausente cuente como "no cumple", y `.copy()` para que `consulta_loc` sea una tabla independiente y se pueda modificar sin afectar a `eva`.

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [5]:
# Desarrollo de la actividad 3
codigo_objetivo = "15"
anios_objetivo = [2023, 2024]

consulta_query = eva.query(
    "codigo_departamento == @codigo_objetivo "
    "and anio.isin(@anios_objetivo) "
    "and area_ha > 0 "
    "and produccion_t >= 0",
    engine="python",
).copy()

pd.testing.assert_frame_equal(consulta_loc, consulta_query)
print("Equivalentes: mismas filas, columnas, índices y valores.")
print("consulta_loc:", consulta_loc.shape, "| consulta_query:", consulta_query.shape)

Equivalentes: mismas filas, columnas, índices y valores.
consulta_loc: (4924, 22) | consulta_query: (4924, 22)


**Interpretación del resultado:**

`consulta_query` expresa la misma selección como texto, usando las variables `@codigo_objetivo` (`"15"`) y `@anios_objetivo` (`[2023, 2024]`). `pd.testing.assert_frame_equal` no lanzó ningún error, lo que confirma que ambas consultas devuelven **las mismas 4.924 filas, en el mismo orden, con las mismas 22 columnas, tipos y valores** (las 18 originales más las 4 de trabajo).

`.loc` con máscaras y `.query` son dos sintaxis para la misma operación lógica. `.query` es más legible cuando hay muchas condiciones; `.loc` es más explícito y permite reutilizar cada condición por separado, como se hizo en la actividad 2 para desglosar las 394 exclusiones por área.

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [6]:
# Desarrollo de la actividad 4
resumen = (
    consulta_loc.groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)
resumen["total_anio"] = resumen.groupby("anio", dropna=False)["registros"].transform("sum")
resumen["porcentaje_anual"] = resumen["registros"] / resumen["total_anio"] * 100
resumen = resumen.sort_values(
    ["anio", "registros", "Cultivo"], ascending=[True, False, True]
).reset_index(drop=True)

# Conciliación
assert resumen["registros"].sum() == len(consulta_loc)
suma_pct = resumen.groupby("anio")["porcentaje_anual"].sum()
assert ((suma_pct - 100).abs() < 1e-9).all()
print("Suma de registros:", int(resumen["registros"].sum()), "= len(consulta_loc):", len(consulta_loc))
print("Porcentaje por año:", {int(k): round(float(v), 6) for k, v in suma_pct.items()})

for anio, grupo in resumen.groupby("anio"):
    print(f"\nAño {anio}: {grupo['total_anio'].iloc[0]:,} registros, {len(grupo)} cultivos. Top 10:")
    display(grupo.head(10).style.format({"porcentaje_anual": "{:.2f} %"}))

Suma de registros: 4924 = len(consulta_loc): 4924
Porcentaje por año: {2023: 100.0, 2024: 100.0}

Año 2023: 2,436 registros, 96 cultivos. Top 10:


,anio,Cultivo,registros,total_anio,porcentaje_anual
0,2023,Papa,239,2436,9.81 %
1,2023,Maíz,194,2436,7.96 %
2,2023,Arveja,176,2436,7.22 %
3,2023,Frijol,162,2436,6.65 %
4,2023,Tomate,113,2436,4.64 %
5,2023,Cebolla de bulbo,93,2436,3.82 %
6,2023,Yuca,82,2436,3.37 %
7,2023,Caña,63,2436,2.59 %
8,2023,Trigo,58,2436,2.38 %
9,2023,Mora,51,2436,2.09 %



Año 2024: 2,488 registros, 96 cultivos. Top 10:


,anio,Cultivo,registros,total_anio,porcentaje_anual
96,2024,Papa,237,2488,9.53 %
97,2024,Maíz,196,2488,7.88 %
98,2024,Frijol,169,2488,6.79 %
99,2024,Arveja,168,2488,6.75 %
100,2024,Tomate,120,2488,4.82 %
101,2024,Cebolla de bulbo,95,2488,3.82 %
102,2024,Yuca,82,2488,3.30 %
103,2024,Caña,66,2488,2.65 %
104,2024,Trigo,57,2488,2.29 %
105,2024,Aguacate,52,2488,2.09 %


**Interpretación del resultado:**

`resumen` tiene una fila por combinación **año–cultivo**, con `registros`, `total_anio` (calculado con `transform("sum")`, que repite el total del año en cada fila) y `porcentaje_anual`. La suma de `registros` es **4.924**, igual a `len(consulta_loc)`, y los porcentajes suman **100 % en cada año**: no se perdió ni se duplicó ningún registro al agrupar.

En Boyacá se reportan **96 cultivos distintos en cada año**: 2.436 registros en 2023 y 2.488 en 2024.

| Puesto | 2023 | % | 2024 | % |
|---:|---|---:|---|---:|
| 1 | Papa (239) | 9,81 | Papa (237) | 9,53 |
| 2 | Maíz (194) | 7,96 | Maíz (196) | 7,88 |
| 3 | Arveja (176) | 7,22 | Frijol (169) | 6,79 |
| 4 | Frijol (162) | 6,65 | Arveja (168) | 6,75 |
| 5 | Tomate (113) | 4,64 | Tomate (120) | 4,82 |

- **Papa** es el cultivo con más registros en ambos años (cerca del 10 %), seguido de **maíz**.
- Los **5 primeros cultivos concentran el 36,3 % (2023) y el 35,8 % (2024)** de los registros, y los **10 primeros, cerca del 50 %**. La otra mitad se reparte entre unos 86 cultivos, muchos con muy pocos registros.
- El ranking es **muy estable** entre años. Los nueve primeros cultivos son los mismos; solo **arveja y frijol intercambian** el 3.º y 4.º puesto, y en el 10.º lugar **mora** (2023) es reemplazada por **aguacate** (2024).


## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [7]:
# Desarrollo de la actividad 5
tabla_dinamica = resumen.pivot_table(
    index="Cultivo",
    columns="anio",
    values="registros",
    aggfunc="sum",
    fill_value=0,
)
tabla_dinamica["total"] = tabla_dinamica.sum(axis=1)
tabla_dinamica = tabla_dinamica.sort_values("total", ascending=False)
display(tabla_dinamica.head(20))

# Conciliación con resumen
por_anio_pivot = tabla_dinamica.drop(columns="total").sum()
por_anio_resumen = resumen.groupby("anio")["registros"].sum()
assert (por_anio_pivot.values == por_anio_resumen.values).all()
assert tabla_dinamica["total"].sum() == resumen["registros"].sum() == len(consulta_loc)
print("Totales por año (pivot):", {int(k): int(v) for k, v in por_anio_pivot.items()})

ruta = OUT / "resumen_eva.csv"
resumen.to_csv(ruta, index=False, encoding="utf-8-sig")
assert ruta.is_file()
print("Archivo generado:", ruta)

# Datos para las conclusiones
anios = [c for c in tabla_dinamica.columns if c != "total"]
for anio in anios:
    fila = resumen.loc[resumen["anio"].eq(anio)].iloc[0]
    top5 = resumen.loc[resumen["anio"].eq(anio)].head(5)["porcentaje_anual"].sum()
    print(f"{anio}: cultivo con más registros = {fila['Cultivo']} "
          f"({fila['registros']} registros, {fila['porcentaje_anual']:.2f} %); top 5 acumula {top5:.2f} %")
if len(anios) == 2:
    a, b = anios
    solo_a = tabla_dinamica.index[(tabla_dinamica[a] > 0) & (tabla_dinamica[b] == 0)].tolist()
    solo_b = tabla_dinamica.index[(tabla_dinamica[a] == 0) & (tabla_dinamica[b] > 0)].tolist()
    print(f"Cultivos solo en {a}: {solo_a}")
    print(f"Cultivos solo en {b}: {solo_b}")

anio,2023,2024,total
Cultivo,,,
Papa,239,237,476
Maíz,194,196,390
Arveja,176,168,344
Frijol,162,169,331
Tomate,113,120,233
Cebolla de bulbo,93,95,188
Yuca,82,82,164
Caña,63,66,129
Trigo,58,57,115


Totales por año (pivot): {2023: 2436, 2024: 2488}
Archivo generado: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02/resumen_eva.csv
2023: cultivo con más registros = Papa (239 registros, 9.81 %); top 5 acumula 36.29 %
2024: cultivo con más registros = Papa (237 registros, 9.53 %); top 5 acumula 35.77 %
Cultivos solo en 2023: ['Mangostino']
Cultivos solo en 2024: ['Apio']


**Interpretación del resultado:**

La tabla dinámica pone los **cultivos en filas** y los **años en columnas**, con el número de registros en cada celda y `fill_value=0` donde no hay datos. Los totales por columna coinciden con `resumen` (**2.436 en 2023 y 2.488 en 2024**) y el gran total con `len(consulta_loc)` (4.924). `resumen` se exportó sin índice a `kit/salidas/soluciones/PD02/resumen_eva.csv`.

**Conclusión 1. Pocos cultivos concentran el reporte y el orden es estable.** **Papa** es el cultivo con más registros en Boyacá en ambos años: 239 en 2023 (9,81 %) y 237 en 2024 (9,53 %). Los cinco primeros cultivos (papa, maíz, arveja, frijol y tomate) acumulan el **36,29 %** de los registros en 2023 y el **35,77 %** en 2024. Los otros ~91 cultivos se reparten el resto, la mayoría con muy pocos registros. Como el ranking casi no cambia entre años, la estructura de **reporte** agrícola del departamento es estable. Esto mide en cuántos municipios y periodos se reportó cada cultivo, **no** su participación en el área sembrada ni en la producción: un cultivo con pocos registros puede abarcar más hectáreas.

**Conclusión 2. Los ceros de la tabla son ausencia de filas, no producción cero.** De los 97 cultivos que aparecen en algún año, 95 se reportan en ambos. **Mangostino** aparece solo en 2023 y **Apio** solo en 2024, y por eso cada uno tiene un 0 en el año en que no aparece. Ese cero significa que **no hubo ninguna fila** de ese cultivo en Boyacá ese año con área cosechada positiva. Puede deberse a que no se reportó, a que se reportó con área cosechada 0 (excluido en la actividad 2) o a un cambio en el registro de la fuente. No demuestra que la producción haya sido cero.

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.